# TrendWaveletAE v2 Tourism-Yearly Study

**Study design:** Successive halving search over TrendWaveletAE and TrendWaveletAELG unified blocks (10 stacks, shared weights) on Tourism-Yearly.

**Search space:** 5 wavelet families (haar, db3, db20, coif2, sym10) x 2 basis labels (eq_fcast, lt_fcast) x 2 latent dims (8, 12) x 2 block types = 40 configs.

**Key questions:**
1. Does any config beat the prior Tourism-Yearly SOTA of SMAPE=20.864?
2. Does TrendWaveletAE outperform TrendWaveletAELG on Tourism (reversing the usual AELG advantage)?
3. Which wavelet family works best for short-horizon Tourism forecasting?
4. Does latent dimension matter at this scale?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.1)

df = pd.read_csv('../../../experiments/results/tourism/trendwaveletae_v2_study_results.csv')
r2 = df[df['search_round'] == 2].copy()
r1 = df[df['search_round'] == 1].copy()

print(f'Total rows: {len(df)} (R1: {len(r1)}, R2: {len(r2)})')
print(f'R2 configs: {r2["config_name"].nunique()}, runs per config: {r2.groupby("config_name")["run"].count().iloc[0]}')
print(f'Diverged: {r2["diverged"].sum()}, SMAPE range: {r2["smape"].min():.3f} - {r2["smape"].max():.3f}')
print(f'Convergence: {r2["stopping_reason"].value_counts().to_dict()}')
print(f'Mean epochs: {r2["epochs_trained"].mean():.1f}')

## Final Rankings (Round 2)

All 20 surviving configs with 5 runs each at up to 50 epochs. The top 3 are separated by only 0.04 SMAPE -- essentially a dead heat.

In [ ]:
config_stats = r2.groupby('config_name').agg(
    smape_mean=('smape', 'mean'),
    smape_std=('smape', 'std'),
    smape_median=('smape', 'median'),
    smape_min=('smape', 'min'),
    smape_max=('smape', 'max'),
    n_params=('n_params', 'first'),
    block_type=('block_type', 'first'),
    wavelet_family=('wavelet_family', 'first'),
    bd_label=('bd_label', 'first'),
    latent_dim=('latent_dim_cfg', 'first'),
).sort_values('smape_mean')

best_smape = config_stats['smape_mean'].iloc[0]
config_stats['delta'] = config_stats['smape_mean'] - best_smape
config_stats['delta_pct'] = config_stats['delta'] / best_smape * 100

display_cols = ['smape_mean', 'smape_std', 'smape_median', 'block_type', 'wavelet_family', 'bd_label', 'latent_dim', 'delta']
config_stats[display_cols].head(20).style.format({
    'smape_mean': '{:.3f}', 'smape_std': '{:.3f}', 'smape_median': '{:.3f}', 'delta': '+{:.3f}'
}).background_gradient(subset=['smape_mean'], cmap='RdYlGn_r')

## Comparison to Prior SOTA (SMAPE = 20.864)

The prior Tourism-Yearly SOTA for TrendWavelet architectures is 20.864 (TrendWaveletAELG coif3_eq_bcast_td3_ld16, 10-run confirmation with 95% CI [20.732, 20.996]).

**The best config in this study (21.013) does NOT beat the prior SOTA.** It is 0.149 SMAPE points worse (+0.72%). Only 1 of 5 runs dipped below 20.864. A one-sample t-test rejects the hypothesis that the mean equals the SOTA (t=2.33, p=0.080), but in the wrong direction -- the mean is higher.

In [ ]:
prior_sota = 20.864

fig, ax = plt.subplots(figsize=(12, 6))
top10 = config_stats.head(10)
x = range(len(top10))
labels = [n.replace('TrendWavelet', 'TW') for n in top10.index]
colors = ['#2196F3' if bt == 'TrendWaveletAE' else '#FF9800' for bt in top10['block_type']]

ax.bar(x, top10['smape_mean'], yerr=top10['smape_std'], capsize=4, color=colors, alpha=0.8)
ax.axhline(y=prior_sota, color='red', linestyle='--', linewidth=2, label=f'Prior SOTA = {prior_sota}')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('SMAPE')
ax.set_title('Top 10 Configs vs Prior Tourism-Yearly SOTA')
ax.legend()

# Custom legend for block types
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='TrendWaveletAE'),
    Patch(facecolor='#FF9800', label='TrendWaveletAELG'),
    plt.Line2D([0], [0], color='red', linestyle='--', label=f'Prior SOTA ({prior_sota})')
]
ax.legend(handles=legend_elements, loc='upper left')
ax.set_ylim(20.5, 22.0)
plt.tight_layout()
plt.show()

## Factor Analysis

### Wavelet Family: Haar is the clear winner

Unlike M4-Yearly where wavelet choice is a non-factor, on Tourism-Yearly **Haar significantly outperforms all other families** (KW p=0.009). This makes sense: Tourism-Yearly has forecast_length=4, and Haar (the simplest, shortest-support wavelet) is naturally well-suited to very short horizons. Longer-support wavelets (db20, sym10) have more coefficients than there are forecast steps, collapsing to approximation-only bases.

In [ ]:
wavelet_data = r2.groupby('wavelet_family')['smape'].agg(['mean', 'std', 'median', 'count']).sort_values('mean')
print(wavelet_data.round(3))

# Kruskal-Wallis
groups = [r2[r2['wavelet_family'] == wf]['smape'].values for wf in wavelet_data.index]
kw = stats.kruskal(*groups)
print(f'\nKruskal-Wallis: H={kw.statistic:.2f}, p={kw.pvalue:.4f}')

# Post-hoc: Haar vs each
haar_vals = r2[r2['wavelet_family'] == 'haar']['smape'].values
print('\nPost-hoc (Haar vs each):')
for wf in ['db3', 'db20', 'sym10', 'coif2']:
    wf_vals = r2[r2['wavelet_family'] == wf]['smape'].values
    u, p = stats.mannwhitneyu(haar_vals, wf_vals)
    d = (haar_vals.mean() - wf_vals.mean()) / np.sqrt((haar_vals.std()**2 + wf_vals.std()**2) / 2)
    print(f'  haar vs {wf}: p={p:.4f}, d={d:.2f} {"*" if p < 0.05 else ""}')

# Box plot
fig, ax = plt.subplots(figsize=(8, 5))
order = wavelet_data.index.tolist()
sns.boxplot(data=r2, x='wavelet_family', y='smape', order=order, ax=ax, palette='Set2')
ax.set_title('SMAPE by Wavelet Family (Round 2)')
ax.set_ylabel('SMAPE')
plt.tight_layout()
plt.show()

### Basis Label: eq_fcast vs lt_fcast -- No Difference

On Tourism-Yearly, eq_fcast gives basis_dim=4 (=forecast_length) and lt_fcast gives basis_dim=2. Despite this 2x difference in basis dimensionality, there is **no significant difference** in SMAPE (MWU p=0.65). This is surprising because lt_fcast (bd=2) is a much more aggressive compression, but apparently the wavelet bases at dim=2 still capture enough structure for 4-step forecasts.

Note: this contrasts with M4-Yearly where lt_fcast is reliably harmful. The difference is likely that Tourism-Yearly's forecast horizon (4) is so short that even 2 basis functions suffice.

In [ ]:
for label in ['eq_fcast', 'lt_fcast']:
    vals = r2[r2['bd_label'] == label]['smape']
    bd = r2[r2['bd_label'] == label]['basis_dim'].iloc[0]
    print(f'{label} (bd={bd}): mean={vals.mean():.3f} +/-{vals.std():.3f}, median={vals.median():.3f}, n={len(vals)}')

u, p = stats.mannwhitneyu(
    r2[r2['bd_label'] == 'eq_fcast']['smape'],
    r2[r2['bd_label'] == 'lt_fcast']['smape']
)
print(f'MWU p={p:.4f} -- No significant difference')

### Latent Dimension: ld=12 significantly better than ld=8

ld=12 outperforms ld=8 (MWU p=0.020, d=0.40 -- small-to-medium effect). The advantage is more pronounced for AELG (0.26 SMAPE gap) than AE (0.07 gap), suggesting the learned gate benefits from a wider latent space to select from.

This is an interesting contrast with M4-Yearly where ld=16 > ld=8 was also found. The pattern is consistent: for TrendWavelet unified blocks, larger latent dimensions help.

In [ ]:
for ld in [8, 12]:
    vals = r2[r2['latent_dim_cfg'] == ld]['smape']
    print(f'ld={ld}: mean={vals.mean():.3f} +/-{vals.std():.3f}, n={len(vals)}')

u, p = stats.mannwhitneyu(
    r2[r2['latent_dim_cfg'] == 8]['smape'],
    r2[r2['latent_dim_cfg'] == 12]['smape']
)
d = (r2[r2['latent_dim_cfg'] == 8]['smape'].mean() - r2[r2['latent_dim_cfg'] == 12]['smape'].mean()) / \
    np.sqrt((r2[r2['latent_dim_cfg'] == 8]['smape'].std()**2 + r2[r2['latent_dim_cfg'] == 12]['smape'].std()**2) / 2)
print(f'MWU p={p:.4f}, Cohen\'s d={d:.3f}')

# Interaction: block_type x latent_dim
print('\nBlock Type x Latent Dim interaction:')
for bt in ['TrendWaveletAE', 'TrendWaveletAELG']:
    for ld in [8, 12]:
        vals = r2[(r2['block_type'] == bt) & (r2['latent_dim_cfg'] == ld)]['smape']
        if len(vals) > 0:
            print(f'  {bt.replace("TrendWavelet", "TW")} ld={ld}: mean={vals.mean():.3f} +/-{vals.std():.3f}, n={len(vals)}')

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=r2, x='latent_dim_cfg', y='smape', hue='block_type', ax=ax)
ax.set_title('SMAPE by Latent Dim and Block Type')
ax.set_xlabel('Latent Dimension')
ax.set_ylabel('SMAPE')
plt.tight_layout()
plt.show()

### Block Type: TrendWaveletAE vs TrendWaveletAELG

**AE edges ahead of AELG but not significantly** (MWU p=0.24, d=0.32). AE occupies 3 of the top 4 positions. The learned gate adds marginal parameters (~120 more) but does not help -- and may slightly hurt by introducing an additional optimization target on this short-horizon task.

This is consistent with the prior finding that AE tends to be more stable than AELG on Tourism. The learned gate's benefit is more apparent on longer horizons (M4-Yearly H=6, Traffic H=96) where there is more latent structure to discover.

In [ ]:
for bt in ['TrendWaveletAE', 'TrendWaveletAELG']:
    vals = r2[r2['block_type'] == bt]['smape']
    print(f'{bt}: mean={vals.mean():.3f} +/-{vals.std():.3f}, median={vals.median():.3f}, n={len(vals)}')

u, p = stats.mannwhitneyu(
    r2[r2['block_type'] == 'TrendWaveletAE']['smape'],
    r2[r2['block_type'] == 'TrendWaveletAELG']['smape']
)
print(f'MWU p={p:.4f}')

## Summary

### Key Findings

1. **No new SOTA.** The best config (TrendWaveletAE_db20_eq_fcast_td3_ld12, SMAPE=21.013) is 0.15 points above the prior SOTA of 20.864. The prior SOTA used coif3 with ld=16 -- a wavelet family and latent dimension not tested in this study.

2. **Haar is the best wavelet for Tourism-Yearly** (KW p=0.009). Short-support wavelets naturally suit short horizons (H=4). This is a dataset-specific finding -- on M4-Yearly (H=6) wavelet family is a non-factor.

3. **ld=12 > ld=8** (MWU p=0.020). Combined with the prior finding that ld=16 > ld=8 on M4, larger latent dims consistently help for unified TrendWavelet blocks.

4. **Basis label does not matter** on Tourism (p=0.65). Even bd=2 (lt_fcast) works fine for 4-step forecasts.

5. **AE and AELG are statistically equivalent** on Tourism (p=0.24). AE has a slight edge, consistent with the general pattern that AELG's advantage grows with horizon length.

6. **Zero divergence** across all 220 runs. Unified TrendWavelet blocks are extremely stable on Tourism.

### Why the SOTA Was Not Beaten

The prior SOTA used coif3 (not tested here) and ld=16 (not tested here). Given that ld=12 > ld=8 was significant, ld=16 likely provides further improvement. The winning wavelet here (db20) is a long-support wavelet that collapses to approximation-only on H=4 -- it effectively becomes a smoothed identity basis, not a true wavelet decomposition.

### Recommendations

1. **Test Haar at ld=16** on Tourism-Yearly. Haar won the wavelet comparison and ld=16 is the established best latent dim. This combination was not tested.
2. **Test coif3 at ld=12** to bridge the gap to the prior SOTA config.
3. **Consider alternating Trend+Wavelet** (not unified TrendWavelet) with Haar, which achieved SMAPE=13.496 on M4-Yearly with only 965K params in the trend_seas_wav comparison study.
4. **Do not invest further in bd_label tuning** for Tourism -- it is a non-factor at this horizon.